<a href="https://colab.research.google.com/github/ralphhurl/kling-jwt/blob/main/4Acre_Trumbull_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests

In [2]:
"""
Trumbull County, Ohio — 4+ Acre Parcel Query
Targets: Vienna Township, Hubbard Township, Fowler Township
Output: CSV with owner name, address, parcel number, acreage

Queries the Trumbull County ArcGIS REST API directly.
Run: python3 trumbull_parcel_query.py

Requirements: pip install requests
"""

import requests
import csv
import json
import sys
import time
from datetime import datetime

# ── CONFIG ──────────────────────────────────────────────────────────────────
MIN_ACRES = 4.0
TARGET_TOWNSHIPS = ["VIENNA", "HUBBARD", "FOWLER"]  # adjust case if needed
OUTPUT_FILE = f"trumbull_4acre_parcels_{datetime.now().strftime('%Y%m%d')}.csv"

# ── CANDIDATE ENDPOINTS ──────────────────────────────────────────────────────
# The script tries each in order; first one that responds wins.
ENDPOINTS = [
    # Trumbull County's own GIS server
    "https://webgis.co.trumbull.oh.us/gisserver/rest/services/Maps/TaxParcels/MapServer/0",
    # Trumbull County open data (ArcGIS Online hosted)
    "https://services1.arcgis.com/gZ4ohrz1BHSL72Wn/arcgis/rest/services/TaxParcels/FeatureServer/0",
    # Ohio DNR statewide parcel layer (fallback — has less detail)
    "https://gis.ohiodnr.gov/arcgis_site2/rest/services/OIT_Services/odnr_landbase_v2/MapServer/4",
]

# ── FIELD MAPPING ────────────────────────────────────────────────────────────
# Each endpoint may use different field names. Define candidates; script probes.
FIELD_CANDIDATES = {
    "parcel":    ["PARCEL_NO", "PIN", "PARCELID", "PARCELNUMBER", "PARID", "STATEWIDE_PIN", "APN"],
    "owner":     ["OWNER", "OWNER1", "OWNERNAME", "OWNER_NAME", "GRANTEE"],
    "address":   ["ADDRESS", "SITEADDRESS", "SITE_ADDR", "ADDR", "PROPERTY_ADDRESS", "AUD_LINK"],
    "township":  ["TOWNSHIP", "TWP", "TAXING_DISTRICT", "POLITICAL_TWP"],
    "acres":     ["ACRES", "ASSR_ACRES", "CALC_ACRES", "ACREAGE", "GIS_ACRES", "DEED_ACRES"],
}

RECORDS_PER_PAGE = 1000  # ArcGIS default max; server may cap lower
MAX_PAGES = 50            # safety ceiling (~50k records)


def probe_fields(base_url: str, session: requests.Session) -> dict:
    """Query first record to discover actual field names."""
    r = session.get(f"{base_url}/query", params={
        "where": "1=1", "outFields": "*", "resultRecordCount": 1, "f": "json"
    }, timeout=15)
    r.raise_for_status()
    data = r.json()

    if "error" in data:
        raise RuntimeError(f"API error: {data['error']}")

    actual_fields = {f["name"].upper(): f["name"]
                     for f in data.get("fields", [])}

    resolved = {}
    for role, candidates in FIELD_CANDIDATES.items():
        for c in candidates:
            if c.upper() in actual_fields:
                resolved[role] = actual_fields[c.upper()]
                break
        if role not in resolved:
            resolved[role] = None  # not found; will be blank in output

    print(f"  Field mapping: {resolved}")
    return resolved


def build_where(fields: dict) -> str:
    """Build WHERE clause for township + acreage filter."""
    township_field = fields.get("township")
    acres_field    = fields.get("acres")

    clauses = []

    if township_field:
        twp_list = ", ".join(f"'{t}'" for t in TARGET_TOWNSHIPS)
        clauses.append(f"UPPER({township_field}) IN ({twp_list})")

    if acres_field:
        clauses.append(f"{acres_field} >= {MIN_ACRES}")

    if not clauses:
        # Fallback: no spatial filter — will pull everything; user must filter CSV
        print("  WARNING: Could not resolve township or acreage fields. Fetching all records.")
        return "1=1"

    return " AND ".join(clauses)


def fetch_all(base_url: str, fields: dict, session: requests.Session) -> list[dict]:
    """Page through all matching records using resultOffset."""
    where = build_where(fields)
    print(f"  WHERE: {where}")

    out_fields = ",".join(v for v in fields.values() if v)
    records = []

    for page in range(MAX_PAGES):
        offset = page * RECORDS_PER_PAGE
        params = {
            "where":             where,
            "outFields":         out_fields,
            "resultRecordCount": RECORDS_PER_PAGE,
            "resultOffset":      offset,
            "returnGeometry":    "false",
            "f":                 "json",
        }
        r = session.get(f"{base_url}/query", params=params, timeout=30)
        r.raise_for_status()
        data = r.json()

        if "error" in data:
            raise RuntimeError(f"API error at offset {offset}: {data['error']}")

        features = data.get("features", [])
        if not features:
            break

        for feat in features:
            attrs = feat.get("attributes", {})
            records.append({
                "parcel":   attrs.get(fields["parcel"])   if fields["parcel"]   else "",
                "owner":    attrs.get(fields["owner"])    if fields["owner"]    else "",
                "address":  attrs.get(fields["address"])  if fields["address"]  else "",
                "township": attrs.get(fields["township"]) if fields["township"] else "",
                "acres":    attrs.get(fields["acres"])    if fields["acres"]    else "",
            })

        print(f"  Page {page+1}: fetched {len(features)} records (total so far: {len(records)})")

        # ArcGIS signals "no more data" via exceededTransferLimit=false after last page
        if not data.get("exceededTransferLimit", False):
            break

        time.sleep(0.2)  # polite pacing

    return records


def write_csv(records: list[dict], filepath: str):
    fieldnames = ["parcel", "owner", "address", "township", "acres"]
    with open(filepath, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(records)
    print(f"\n  Wrote {len(records)} records → {filepath}")


def main():
    session = requests.Session()
    session.headers.update({"User-Agent": "Trumbull-Parcel-Query/1.0"})

    for endpoint in ENDPOINTS:
        print(f"\nTrying endpoint: {endpoint}")
        try:
            fields = probe_fields(endpoint, session)
            records = fetch_all(endpoint, fields, session)

            # Post-filter by acreage if field wasn't in WHERE (e.g. string field edge case)
            records = [r for r in records
                       if r["acres"] == "" or float(str(r["acres"]) or 0) >= MIN_ACRES]

            # Post-filter by township if field resolution was ambiguous
            records = [r for r in records
                       if not r["township"]
                       or any(t in str(r["township"]).upper() for t in TARGET_TOWNSHIPS)]

            if not records:
                print("  No matching records from this endpoint — trying next.")
                continue

            write_csv(records, OUTPUT_FILE)

            print("\n── SAMPLE (first 10 rows) ──")
            for r in records[:10]:
                print(f"  {r['parcel']:<20} {str(r['acres']):<8} ac  {r['owner']:<35} {r['address']}")

            print(f"\nDone. {len(records)} parcels written to: {OUTPUT_FILE}")
            return

        except Exception as e:
            print(f"  Failed: {e}")
            continue

    print("\nAll endpoints failed. See fallback instructions below.")
    print_fallback()


def print_fallback():
    print("""
── FALLBACK: Manual Download ──────────────────────────────────────────────────

1. AcreValue (free, no account):
   https://www.acrevalue.com/plat-map/OH/Trumbull/
   → Click Vienna / Hubbard / Fowler Township
   → Pan/zoom; click parcels for owner + acreage
   → No bulk export without paid plan

2. Regrid (free tier — 25 lookups/day; bulk export requires Pro ~$99/yr):
   https://app.regrid.com/us/oh/trumbull
   → Use Filter: Acres >= 4, Township IN [Vienna, Hubbard, Fowler]
   → Export CSV (Pro) or copy manually (free)

3. Ohio DNR Open Data (direct download, free):
   https://ohiodnr.gov/discover-and-learn/safety-conservation/land/ohio-statewide-land-cover
   → Search: "Ohio Statewide Parcels"
   → Download GIS shapefile → open in QGIS (free) → filter + export CSV

4. Trumbull County Auditor direct contact:
   (330) 675-2420
   Request a data extract of parcels ≥4 acres in Vienna, Hubbard, Fowler townships.
   Ohio Public Records law (ORC 149.43) requires they provide it; usually ~$25-50 flat fee.
""")


if __name__ == "__main__":
    main()



Trying endpoint: https://webgis.co.trumbull.oh.us/gisserver/rest/services/Maps/TaxParcels/MapServer/0
  Field mapping: {'parcel': 'PIN', 'owner': None, 'address': None, 'township': None, 'acres': None}
  WHERE: 1=1
  Page 1: fetched 1000 records (total so far: 1000)
  Page 2: fetched 1000 records (total so far: 2000)
  Page 3: fetched 1000 records (total so far: 3000)
  Page 4: fetched 1000 records (total so far: 4000)
  Page 5: fetched 1000 records (total so far: 5000)
  Page 6: fetched 1000 records (total so far: 6000)
  Page 7: fetched 1000 records (total so far: 7000)
  Page 8: fetched 1000 records (total so far: 8000)
  Page 9: fetched 1000 records (total so far: 9000)
  Page 10: fetched 1000 records (total so far: 10000)
  Page 11: fetched 1000 records (total so far: 11000)
  Page 12: fetched 1000 records (total so far: 12000)
  Page 13: fetched 1000 records (total so far: 13000)
  Page 14: fetched 1000 records (total so far: 14000)
  Page 15: fetched 1000 records (total so far:

In [3]:
import requests, json

url = "https://webgis.co.trumbull.oh.us/gisserver/rest/services/Maps/TaxParcels/MapServer/0/query"
params = {
    "where": "1=1",
    "outFields": "*",
    "resultRecordCount": 1,
    "f": "json"
}
r = requests.get(url, params=params)
data = r.json()

print("=== FIELDS ===")
for f in data.get("fields", []):
    print(f['name'], "→", f['alias'])

print("\n=== SAMPLE RECORD ===")
if data.get("features"):
    print(json.dumps(data["features"][0]["attributes"], indent=2))

=== FIELDS ===
OBJECTID → OBJECTID
PIN → PIN
ROUTING → ROUTING
PARCEL_ID → PARCEL_ID
INSTRUMENT_NUM → INSTRUMENT_NUM
PARCEL_FIXED → PARCEL_FIXED
OWNERSHIP_OID → OWNERSHIP_OID
OWNERSHIP_COUNT → OWNERSHIP_COUNT
EditedBy → EditedBy
CreatedBy → CreatedBy
EditedByDate → EditedByDate
CreatedByDate → CreatedByDate
mpropertyNumber → mpropertyNumber
mmapNumber → mmapNumber
mcamaId → mcamaId
CardCount → CardCount
NeighborhoodCode → NeighborhoodCode
mlegalDescription → mlegalDescription
mlocStrDir → mlocStrDir
mlocStrNo → mlocStrNo
mlocStrNo2 → mlocStrNo2
mlocStrName → mlocStrName
mlocStrSuffix → mlocStrSuffix
mlocStrSuffixDir → mlocStrSuffixDir
msecondaryAddress → msecondaryAddress
mlocCity → mlocCity
mlocState → mlocState
mlocZipCode → mlocZipCode
mlocDescription → mlocDescription
mClassificationId → mClassificationId
macres → macres
mfrontfootage → mfrontfootage
mdeedVolume → mdeedVolume
mdeedPage → mdeedPage
Par_Deleted → Par_Deleted
DeededOwner → DeededOwner
mlot → mlot
msublot → msublot
Own

In [4]:
import requests, csv, time

BASE_URL = "https://webgis.co.trumbull.oh.us/gisserver/rest/services/Maps/TaxParcels/MapServer/0/query"
OUTPUT   = "trumbull_4acre_parcels_20260410.csv"

OUT_FIELDS = ",".join([
    "PARCEL_ID", "OwnName", "OwnBusiness",
    "mlocStrNo", "mlocStrName", "mlocStrSuffix", "mlocCity",
    "macres", "NeighborhoodCode", "ROUTING",
    "mlegalDescription", "Par_Deleted",
    "MKT_Total_Value", "AnnualTax", "Zoning",
    "TaxpStreetNumber","TaxpStreetName","TaxpStreetSuffix","TaxpCity","TaxpState","TaxpZipcode"
])

# Township neighborhood code prefixes for Vienna=406, Hubbard=407, Fowler=408
# (based on sample: Warren city = 40600 — we'll pull all 4+ acre and include the code so you can verify)
WHERE = "macres >= 4 AND Par_Deleted = 'N'"

records = []
page = 0

while True:
    params = {
        "where":             WHERE,
        "outFields":         OUT_FIELDS,
        "resultRecordCount": 1000,
        "resultOffset":      page * 1000,
        "returnGeometry":    "false",
        "f":                 "json"
    }
    r = requests.get(BASE_URL, params=params, timeout=30)
    data = r.json()

    features = data.get("features", [])
    if not features:
        break

    for feat in features:
        a = feat["attributes"]
        # Build property address
        prop_addr = " ".join(filter(None, [
            str(a.get("mlocStrNo") or "").strip(),
            str(a.get("mlocStrName") or "").strip(),
            str(a.get("mlocStrSuffix") or "").strip(),
            str(a.get("mlocCity") or "").strip(),
        ]))
        # Build owner mailing address
        mail_addr = " ".join(filter(None, [
            str(a.get("TaxpStreetNumber") or "").strip(),
            str(a.get("TaxpStreetName") or "").strip(),
            str(a.get("TaxpStreetSuffix") or "").strip(),
            str(a.get("TaxpCity") or "").strip(),
            str(a.get("TaxpState") or "").strip(),
            str(a.get("TaxpZipcode") or "").strip(),
        ]))
        records.append({
            "parcel_id":       a.get("PARCEL_ID"),
            "owner_name":      a.get("OwnName"),
            "is_business":     a.get("OwnBusiness"),
            "property_addr":   prop_addr,
            "mailing_addr":    mail_addr,
            "acres":           a.get("macres"),
            "market_value":    a.get("MKT_Total_Value"),
            "annual_tax":      a.get("AnnualTax"),
            "zoning":          a.get("Zoning"),
            "neighborhood_cd": a.get("NeighborhoodCode","").strip(),
            "routing":         a.get("ROUTING","").strip(),
            "legal_desc":      str(a.get("mlegalDescription") or "").strip(),
        })

    print(f"Page {page+1}: {len(features)} records | Total: {len(records)}")
    if not data.get("exceededTransferLimit", False):
        break
    page += 1
    time.sleep(0.2)

# Write CSV
with open(OUTPUT, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=records[0].keys())
    writer.writeheader()
    writer.writerows(records)

print(f"\nDone. {len(records)} parcels → {OUTPUT}")
print("\nSAMPLE (first 5):")
for r in records[:5]:
    print(f"  {r['parcel_id']}  {r['acres']} ac  {r['owner_name']}  NC:{r['neighborhood_cd']}")

Page 1: 1000 records | Total: 1000
Page 2: 1000 records | Total: 2000
Page 3: 1000 records | Total: 3000
Page 4: 1000 records | Total: 4000
Page 5: 1000 records | Total: 5000
Page 6: 1000 records | Total: 6000
Page 7: 1000 records | Total: 7000
Page 8: 1000 records | Total: 8000
Page 9: 1000 records | Total: 9000
Page 10: 1000 records | Total: 10000
Page 11: 1000 records | Total: 11000


AttributeError: 'NoneType' object has no attribute 'strip'

In [5]:
import requests, csv, time

BASE_URL = "https://webgis.co.trumbull.oh.us/gisserver/rest/services/Maps/TaxParcels/MapServer/0/query"
OUTPUT   = "trumbull_4acre_parcels_20260410.csv"

OUT_FIELDS = ",".join([
    "PARCEL_ID", "OwnName", "OwnBusiness",
    "mlocStrNo", "mlocStrName", "mlocStrSuffix", "mlocCity",
    "macres", "NeighborhoodCode", "ROUTING",
    "mlegalDescription", "Par_Deleted",
    "MKT_Total_Value", "AnnualTax", "Zoning",
    "TaxpStreetNumber","TaxpStreetName","TaxpStreetSuffix",
    "TaxpCity","TaxpState","TaxpZipcode"
])

WHERE = "macres >= 4 AND Par_Deleted = 'N'"

def safe(val):
    return str(val).strip() if val is not None else ""

records = []
page = 0

while True:
    params = {
        "where":             WHERE,
        "outFields":         OUT_FIELDS,
        "resultRecordCount": 1000,
        "resultOffset":      page * 1000,
        "returnGeometry":    "false",
        "f":                 "json"
    }
    r = requests.get(BASE_URL, params=params, timeout=30)
    data = r.json()
    features = data.get("features", [])
    if not features:
        break

    for feat in features:
        a = feat["attributes"]
        prop_addr = " ".join(filter(None, [
            safe(a.get("mlocStrNo")), safe(a.get("mlocStrName")),
            safe(a.get("mlocStrSuffix")), safe(a.get("mlocCity")),
        ]))
        mail_addr = " ".join(filter(None, [
            safe(a.get("TaxpStreetNumber")), safe(a.get("TaxpStreetName")),
            safe(a.get("TaxpStreetSuffix")), safe(a.get("TaxpCity")),
            safe(a.get("TaxpState")), safe(a.get("TaxpZipcode")),
        ]))
        records.append({
            "parcel_id":       safe(a.get("PARCEL_ID")),
            "owner_name":      safe(a.get("OwnName")),
            "is_business":     safe(a.get("OwnBusiness")),
            "property_addr":   prop_addr,
            "mailing_addr":    mail_addr,
            "acres":           a.get("macres"),
            "market_value":    a.get("MKT_Total_Value"),
            "annual_tax":      a.get("AnnualTax"),
            "zoning":          safe(a.get("Zoning")),
            "neighborhood_cd": safe(a.get("NeighborhoodCode")),
            "routing":         safe(a.get("ROUTING")),
            "legal_desc":      safe(a.get("mlegalDescription")),
        })

    print(f"Page {page+1}: {len(features)} records | Total: {len(records)}")
    if not data.get("exceededTransferLimit", False):
        break
    page += 1
    time.sleep(0.2)

with open(OUTPUT, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=records[0].keys())
    writer.writeheader()
    writer.writerows(records)

print(f"\nDone. {len(records)} parcels → {OUTPUT}")
print("\nSAMPLE (first 5):")
for r in records[:5]:
    print(f"  {r['parcel_id']}  {r['acres']} ac  {r['owner_name']}  NC:{r['neighborhood_cd']}")

Page 1: 1000 records | Total: 1000
Page 2: 1000 records | Total: 2000
Page 3: 1000 records | Total: 3000
Page 4: 1000 records | Total: 4000
Page 5: 1000 records | Total: 5000
Page 6: 1000 records | Total: 6000
Page 7: 1000 records | Total: 7000
Page 8: 1000 records | Total: 8000
Page 9: 1000 records | Total: 9000
Page 10: 1000 records | Total: 10000
Page 11: 1000 records | Total: 11000
Page 12: 1000 records | Total: 12000
Page 13: 1000 records | Total: 13000
Page 14: 1000 records | Total: 14000
Page 15: 3 records | Total: 14003

Done. 14003 parcels → trumbull_4acre_parcels_20260410.csv

SAMPLE (first 5):
  39-522898  4.97 ac  SARSON HOLDINGS LLC  NC:41001
  39-524520  4.8105 ac  DORIAN CAPITAL  NC:41403
  39-530985  28.0681 ac  P AND M SAWMILL LLC  NC:41401
  39-530985  28.0681 ac  P AND M SAWMILL LLC  NC:41401
  39-530985  28.0681 ac  P AND M SAWMILL LLC  NC:41401


In [6]:
import requests, json

BASE_URL = "https://webgis.co.trumbull.oh.us/gisserver/rest/services/Maps/TaxParcels/MapServer/0/query"

# Known addresses anchored to each target township
probes = {
    "FOWLER TWP":  "3364 YOUNGSTOWN KINGSVILLE",   # Township zoning office road
    "HUBBARD TWP": "2600 ELMWOOD",                  # Township zoning office road
    "VIENNA TWP":  "2577 WARREN SHARON",            # SR-82 through Vienna Twp center
}

for township, street in probes.items():
    params = {
        "where":             f"UPPER(mlocStrName) LIKE '%{street.split()[0]}%'",
        "outFields":         "PARCEL_ID,OwnName,mlocStrNo,mlocStrName,mlocCity,macres,NeighborhoodCode,ROUTING",
        "resultRecordCount": 3,
        "returnGeometry":    "false",
        "f":                 "json"
    }
    r = requests.get(BASE_URL, params=params, timeout=15)
    features = r.json().get("features", [])
    print(f"\n── {township} (searching '{street}') ──")
    for feat in features:
        a = feat["attributes"]
        print(f"  NC:{a.get('NeighborhoodCode','').strip()}  "
              f"ROUTING:{a.get('ROUTING','').strip()}  "
              f"ADDR:{a.get('mlocStrNo','')} {a.get('mlocStrName','').strip()}  "
              f"CITY:{a.get('mlocCity','') or ''}  "
              f"ACRES:{a.get('macres')}")


── FOWLER TWP (searching '3364 YOUNGSTOWN KINGSVILLE') ──

── HUBBARD TWP (searching '2600 ELMWOOD') ──

── VIENNA TWP (searching '2577 WARREN SHARON') ──


In [7]:
import requests

BASE_URL = "https://webgis.co.trumbull.oh.us/gisserver/rest/services/Maps/TaxParcels/MapServer/0/query"

# Zip codes anchored to each township (not the cities themselves)
probes = [
    ("FOWLER TWP",  "mlocZipCode = '44418'"),           # Fowler, OH zip
    ("HUBBARD TWP", "mlocZipCode = '44425' AND mlocCity IS NULL"),  # rural Hubbard Twp (not city)
    ("VIENNA TWP",  "mlocZipCode = '44473'"),           # Vienna, OH zip
    # Fallback — pull a spread of neighborhood codes from 4+ acre rural parcels
    ("RURAL SAMPLE","macres >= 20 AND Par_Deleted = 'N'"),
]

for label, where in probes:
    params = {
        "where":             where,
        "outFields":         "PARCEL_ID,mlocStrName,mlocCity,mlocZipCode,macres,NeighborhoodCode,ROUTING,mlegalDescription",
        "resultRecordCount": 5,
        "returnGeometry":    "false",
        "f":                 "json"
    }
    r = requests.get(BASE_URL, params=params, timeout=15)
    features = r.json().get("features", [])
    print(f"\n── {label} ──")
    if not features:
        print("  (no results)")
    for feat in features:
        a = feat["attributes"]
        print(f"  NC:{str(a.get('NeighborhoodCode') or '').strip():<8} "
              f"ZIP:{a.get('mlocZipCode') or '':<6} "
              f"CITY:{str(a.get('mlocCity') or '').strip():<15} "
              f"ACRES:{a.get('macres'):<8} "
              f"ROUTING:{str(a.get('ROUTING') or '').strip():<18} "
              f"LEGAL:{str(a.get('mlegalDescription') or '').strip()[:40]}")


── FOWLER TWP ──
  (no results)

── HUBBARD TWP ──
  (no results)

── VIENNA TWP ──
  (no results)

── RURAL SAMPLE ──
  NC:41401    ZIP:       CITY:                ACRES:28.0681  ROUTING:39-013 -25803      LEGAL:13          28.0681A^PTS      RESIDUE^MA
  NC:41401    ZIP:       CITY:                ACRES:28.0681  ROUTING:39-013 -25800      LEGAL:13          28.0681A^PTS      RESIDUE^MA
  NC:41401    ZIP:       CITY:                ACRES:28.0681  ROUTING:39-013 -25803      LEGAL:13          28.0681A^PTS      RESIDUE^MA
  NC:40900    ZIP:       CITY:                ACRES:141.24   ROUTING:40-051B-01600      LEGAL:23&24& 33-22   141.239A  ^30 7-11 PTS   
  NC:40900    ZIP:       CITY:                ACRES:34.2     ROUTING:40-051A-36100      LEGAL:24&   48&12  34.20A      ^25 ALL WRN CIT


In [8]:
import requests

BASE_URL = "https://webgis.co.trumbull.oh.us/gisserver/rest/services/Maps/TaxParcels/MapServer/0/query"

# Search legal descriptions and school district codes for township anchors
probes = [
    ("FOWLER - legal desc",   "mlegalDescription LIKE '%FOWLER%'"),
    ("VIENNA - legal desc",   "mlegalDescription LIKE '%VIENNA%'"),
    ("HUBBARD - legal desc",  "mlegalDescription LIKE '%HUBBARD%'"),
    ("School code 7804",      "mstateSchoolcode = '7804'"),  # Mathews LSD (Fowler area)
    ("School code 7806",      "mstateSchoolcode = '7806'"),  # Hubbard EVSD
    ("School code 7814",      "mstateSchoolcode = '7814'"),  # Lakeview LSD (Vienna area)
    ("School code 7816",      "mstateSchoolcode = '7816'"),  # another candidate
]

for label, where in probes:
    params = {
        "where":             f"({where}) AND macres >= 4 AND Par_Deleted = 'N'",
        "outFields":         "PARCEL_ID,mlocStrName,mlocCity,macres,NeighborhoodCode,ROUTING,mstateSchoolcode,mlegalDescription",
        "resultRecordCount": 4,
        "returnGeometry":    "false",
        "f":                 "json"
    }
    r = requests.get(BASE_URL, params=params, timeout=15)
    features = r.json().get("features", [])
    print(f"\n── {label} ──")
    if not features:
        print("  (no results)")
    for feat in features:
        a = feat["attributes"]
        print(f"  NC:{str(a.get('NeighborhoodCode') or '').strip():<8} "
              f"SCH:{a.get('mstateSchoolcode') or '':<6} "
              f"ACRES:{a.get('macres'):<8} "
              f"ROUTING:{str(a.get('ROUTING') or '').strip():<18} "
              f"LEGAL:{str(a.get('mlegalDescription') or '').strip()[:50]}")


── FOWLER - legal desc ──
  NC:28602    SCH:7806   ACRES:35.6928  ROUTING:11-022 -13801      LEGAL:56      2     35.6928A^PT                 ^FOWLER 
  NC:13000    SCH:7806   ACRES:7.32     ROUTING:16-009 -00600      LEGAL:37    4      307.35F     ^PT 7.32A FOWLER RIDGE RD
  NC:29000    SCH:7806   ACRES:51.02    ROUTING:17-024 -02500      LEGAL:12&   1&4    51.0238A    ^22 PT PER SURVEY YO     
  NC:29000    SCH:7806   ACRES:5.0      ROUTING:17-025 -00600      LEGAL:21-   3-4    279.53F     ^31 IRREG 5.00A          

── VIENNA - legal desc ──
  NC:13000    SCH:7806   ACRES:12.0     ROUTING:16-021 -03200      LEGAL:26    6      396F        ^PT 12A SR 170 VIENNA TWP
  NC:13000    SCH:7806   ACRES:15.534   ROUTING:16-005 -01100      LEGAL:22   4   179.51F   15.534A^       PER SURVEY ^KING
  NC:13000    SCH:7806   ACRES:32.4     ROUTING:16-026 -01200      LEGAL:34    1      32.40A      ^PT NILES VIENNA RD      
  NC:13000    SCH:7806   ACRES:65.0     ROUTING:16-020 -01200      LEGAL:35   

In [9]:
import requests, csv, time

BASE_URL = "https://webgis.co.trumbull.oh.us/gisserver/rest/services/Maps/TaxParcels/MapServer/0/query"
OUTPUT   = "trumbull_4acre_parcels_final.csv"

OUT_FIELDS = ",".join([
    "PARCEL_ID", "OwnName", "OwnBusiness",
    "mlocStrNo", "mlocStrName", "mlocStrSuffix", "mlocCity",
    "macres", "NeighborhoodCode", "ROUTING",
    "mlegalDescription", "Par_Deleted",
    "MKT_Total_Value", "AnnualTax", "Zoning",
    "TaxpStreetNumber","TaxpStreetName","TaxpStreetSuffix",
    "TaxpCity","TaxpState","TaxpZipcode"
])

WHERE = ("macres >= 4 AND Par_Deleted = 'N' AND ("
         "ROUTING LIKE '01-%' OR "
         "ROUTING LIKE '11-%' OR "
         "ROUTING LIKE '16-%' OR "
         "ROUTING LIKE '17-%')")

def safe(val):
    return str(val).strip() if val is not None else ""

records = []
page = 0

while True:
    params = {
        "where":             WHERE,
        "outFields":         OUT_FIELDS,
        "resultRecordCount": 1000,
        "resultOffset":      page * 1000,
        "returnGeometry":    "false",
        "f":                 "json"
    }
    r = requests.get(BASE_URL, params=params, timeout=30)
    data = r.json()
    features = data.get("features", [])
    if not features:
        break

    for feat in features:
        a = feat["attributes"]

        routing = safe(a.get("ROUTING"))
        if routing.startswith("01-"):
            twp = "HUBBARD"
        elif routing.startswith("11-") or routing.startswith("17-"):
            twp = "FOWLER"
        elif routing.startswith("16-"):
            twp = "VIENNA"
        else:
            twp = "UNKNOWN"

        prop_addr = " ".join(filter(None, [
            safe(a.get("mlocStrNo")), safe(a.get("mlocStrName")),
            safe(a.get("mlocStrSuffix")), safe(a.get("mlocCity")),
        ]))
        mail_addr = " ".join(filter(None, [
            safe(a.get("TaxpStreetNumber")), safe(a.get("TaxpStreetName")),
            safe(a.get("TaxpStreetSuffix")), safe(a.get("TaxpCity")),
            safe(a.get("TaxpState")), safe(a.get("TaxpZipcode")),
        ]))

        records.append({
            "township":        twp,
            "parcel_id":       safe(a.get("PARCEL_ID")),
            "owner_name":      safe(a.get("OwnName")),
            "is_business":     safe(a.get("OwnBusiness")),
            "property_addr":   prop_addr,
            "mailing_addr":    mail_addr,
            "acres":           a.get("macres"),
            "market_value":    a.get("MKT_Total_Value"),
            "annual_tax":      a.get("AnnualTax"),
            "zoning":          safe(a.get("Zoning")),
            "neighborhood_cd": safe(a.get("NeighborhoodCode")),
            "routing":         routing,
            "legal_desc":      safe(a.get("mlegalDescription")),
        })

    print(f"Page {page+1}: {len(features)} records | Total so far: {len(records)}")
    if not data.get("exceededTransferLimit", False):
        break
    page += 1
    time.sleep(0.2)

with open(OUTPUT, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "township","parcel_id","owner_name","is_business",
        "property_addr","mailing_addr","acres","market_value","annual_tax",
        "zoning","neighborhood_cd","routing","legal_desc"
    ])
    writer.writeheader()
    writer.writerows(records)

print(f"\nDone. {len(records)} parcels written to: {OUTPUT}")
print(f"\nBreakdown by township:")
for twp in ["HUBBARD","FOWLER","VIENNA","UNKNOWN"]:
    count = sum(1 for r in records if r["township"] == twp)
    if count:
        print(f"  {twp}: {count}")

print("\nSAMPLE (first 10):")
for r in records[:10]:
    print(f"  {r['township']:<8} {r['parcel_id']:<12} {r['acres']:<8} {r['owner_name']:<35} {r['property_addr']}")

Page 1: 1000 records | Total so far: 1000
Page 2: 626 records | Total so far: 1626

Done. 1626 parcels written to: trumbull_4acre_parcels_final.csv

Breakdown by township:
  HUBBARD: 575
  FOWLER: 558
  VIENNA: 493

SAMPLE (first 10):
  HUBBARD  01-005004    50.69    BD OF PK COMM OF HUBBARD            0 ROOSEVELT
  HUBBARD  01-014687    10.0     JOAN FLEN                           4011 S MAIN
  HUBBARD  01-015100    13.85    GERALD RICHARD MARTIN               7909 JOHN WHITE
  HUBBARD  01-019001    4.96     JOSEPH M TAKASH J/S                 1766 KOKOMO DR
  HUBBARD  01-019300    6.96     DANIEL K ELLIOTT JR J/S             0 S MAIN
  HUBBARD  01-020900    4.06     JAMES M STEINER JR                  5811 YOUNGSTOWN HUBBARD
  HUBBARD  01-021700    201.13   WELFAM LTD                          5902 CHESTNUT RIDGE
  HUBBARD  01-030400    5.82     CYNTHIA L KOCHER                    6744 CHESTNUT RIDGE
  HUBBARD  01-036760    4.03     JOHN E BIZUB J/S                    7606 JOHN WHITE


In [10]:
import requests, csv, time

BASE_URL = "https://webgis.co.trumbull.oh.us/gisserver/rest/services/Maps/TaxParcels/MapServer/0/query"
OUTPUT   = "trumbull_4acre_parcels_final.csv"

OUT_FIELDS = ",".join([
    "PARCEL_ID", "OwnName", "OwnBusiness",
    "mlocStrNo", "mlocStrName", "mlocStrSuffix", "mlocCity",
    "macres", "NeighborhoodCode", "ROUTING",
    "mlegalDescription", "Par_Deleted",
    "MKT_Total_Value", "AnnualTax", "Zoning",
    "TaxpStreetNumber","TaxpStreetName","TaxpStreetSuffix",
    "TaxpCity","TaxpState","TaxpZipcode"
])

WHERE = ("macres >= 4 AND Par_Deleted = 'N' AND MKT_Impr_Value = 0 AND ("
         "ROUTING LIKE '01-%' OR "
         "ROUTING LIKE '11-%' OR "
         "ROUTING LIKE '16-%' OR "
         "ROUTING LIKE '17-%')")
def safe(val):
    return str(val).strip() if val is not None else ""

records = []
page = 0

while True:
    params = {
        "where":             WHERE,
        "outFields":         OUT_FIELDS,
        "resultRecordCount": 1000,
        "resultOffset":      page * 1000,
        "returnGeometry":    "false",
        "f":                 "json"
    }
    r = requests.get(BASE_URL, params=params, timeout=30)
    data = r.json()
    features = data.get("features", [])
    if not features:
        break

    for feat in features:
        a = feat["attributes"]

        routing = safe(a.get("ROUTING"))
        if routing.startswith("01-"):
            twp = "HUBBARD"
        elif routing.startswith("11-") or routing.startswith("17-"):
            twp = "FOWLER"
        elif routing.startswith("16-"):
            twp = "VIENNA"
        else:
            twp = "UNKNOWN"

        prop_addr = " ".join(filter(None, [
            safe(a.get("mlocStrNo")), safe(a.get("mlocStrName")),
            safe(a.get("mlocStrSuffix")), safe(a.get("mlocCity")),
        ]))
        mail_addr = " ".join(filter(None, [
            safe(a.get("TaxpStreetNumber")), safe(a.get("TaxpStreetName")),
            safe(a.get("TaxpStreetSuffix")), safe(a.get("TaxpCity")),
            safe(a.get("TaxpState")), safe(a.get("TaxpZipcode")),
        ]))

        records.append({
            "township":        twp,
            "parcel_id":       safe(a.get("PARCEL_ID")),
            "owner_name":      safe(a.get("OwnName")),
            "is_business":     safe(a.get("OwnBusiness")),
            "property_addr":   prop_addr,
            "mailing_addr":    mail_addr,
            "acres":           a.get("macres"),
            "market_value":    a.get("MKT_Total_Value"),
            "annual_tax":      a.get("AnnualTax"),
            "zoning":          safe(a.get("Zoning")),
            "neighborhood_cd": safe(a.get("NeighborhoodCode")),
            "routing":         routing,
            "legal_desc":      safe(a.get("mlegalDescription")),
        })

    print(f"Page {page+1}: {len(features)} records | Total so far: {len(records)}")
    if not data.get("exceededTransferLimit", False):
        break
    page += 1
    time.sleep(0.2)

with open(OUTPUT, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "township","parcel_id","owner_name","is_business",
        "property_addr","mailing_addr","acres","market_value","annual_tax",
        "zoning","neighborhood_cd","routing","legal_desc"
    ])
    writer.writeheader()
    writer.writerows(records)

print(f"\nDone. {len(records)} parcels written to: {OUTPUT}")
print(f"\nBreakdown by township:")
for twp in ["HUBBARD","FOWLER","VIENNA","UNKNOWN"]:
    count = sum(1 for r in records if r["township"] == twp)
    if count:
        print(f"  {twp}: {count}")

print("\nSAMPLE (first 10):")
for r in records[:10]:
    print(f"  {r['township']:<8} {r['parcel_id']:<12} {r['acres']:<8} {r['owner_name']:<35} {r['property_addr']}")

Page 1: 532 records | Total so far: 532

Done. 532 parcels written to: trumbull_4acre_parcels_final.csv

Breakdown by township:
  HUBBARD: 189
  FOWLER: 177
  VIENNA: 166

SAMPLE (first 10):
  HUBBARD  01-019300    6.96     DANIEL K ELLIOTT JR J/S             0 S MAIN
  HUBBARD  01-038900    5.01     TIMOTHY W BLAKE J/S                 2960 SCHOTTEN
  HUBBARD  01-063932    7.94     CAROL SYLVESTER                     0 SEIFERT LEWIS
  HUBBARD  01-066017    26.95    KATHRYN SARISKY J/S                 0 WICK CAMPBELL
  HUBBARD  01-071995    27.3364  BRANKO MARKULIN                     0 WICK CAMPBELL RD
  HUBBARD  01-137800    8.84     EDWARD J YASECHKO                   LONG
  HUBBARD  01-241170    9.86     WILLIAM N PATRICK J/S               E LIBERTY
  HUBBARD  01-259650    76.41    NICHOLAS V LIBEG TR                 4556 CATHERINE
  HUBBARD  01-260220    27.0848  R A A W LLC                         0 CHURCHILL HUBBARD RD
  HUBBARD  01-279500    12.39    MASSULLO FARMS &            

In [11]:
import requests, csv, time

BASE_URL = "https://webgis.co.trumbull.oh.us/gisserver/rest/services/Maps/TaxParcels/MapServer/0/query"
OUTPUT   = "trumbull_4acre_parcels_final-Matt.csv"

OUT_FIELDS = ",".join([
    "PARCEL_ID", "OwnName", "OwnBusiness",
    "mlocStrNo", "mlocStrName", "mlocStrSuffix", "mlocCity",
    "macres", "NeighborhoodCode", "ROUTING",
    "mlegalDescription", "Par_Deleted",
    "MKT_Total_Value", "AnnualTax", "Zoning",
    "TaxpStreetNumber","TaxpStreetName","TaxpStreetSuffix",
    "TaxpCity","TaxpState","TaxpZipcode"
])

WHERE = ("macres >= 4 AND Par_Deleted = 'N' AND MKT_Impr_Value = 0 AND ("
         "ROUTING LIKE '01-%' OR "
         "ROUTING LIKE '11-%' OR "
         "ROUTING LIKE '16-%' OR "
         "ROUTING LIKE '17-%')")
def safe(val):
    return str(val).strip() if val is not None else ""

records = []
page = 0

while True:
    params = {
        "where":             WHERE,
        "outFields":         OUT_FIELDS,
        "resultRecordCount": 1000,
        "resultOffset":      page * 1000,
        "returnGeometry":    "false",
        "f":                 "json"
    }
    r = requests.get(BASE_URL, params=params, timeout=30)
    data = r.json()
    features = data.get("features", [])
    if not features:
        break

    for feat in features:
        a = feat["attributes"]

        routing = safe(a.get("ROUTING"))
        if routing.startswith("01-"):
            twp = "HUBBARD"
        elif routing.startswith("11-") or routing.startswith("17-"):
            twp = "FOWLER"
        elif routing.startswith("16-"):
            twp = "VIENNA"
        else:
            twp = "UNKNOWN"

        prop_addr = " ".join(filter(None, [
            safe(a.get("mlocStrNo")), safe(a.get("mlocStrName")),
            safe(a.get("mlocStrSuffix")), safe(a.get("mlocCity")),
        ]))
        mail_addr = " ".join(filter(None, [
            safe(a.get("TaxpStreetNumber")), safe(a.get("TaxpStreetName")),
            safe(a.get("TaxpStreetSuffix")), safe(a.get("TaxpCity")),
            safe(a.get("TaxpState")), safe(a.get("TaxpZipcode")),
        ]))

        records.append({
            "township":        twp,
            "parcel_id":       safe(a.get("PARCEL_ID")),
            "owner_name":      safe(a.get("OwnName")),
            "is_business":     safe(a.get("OwnBusiness")),
            "property_addr":   prop_addr,
            "mailing_addr":    mail_addr,
            "acres":           a.get("macres"),
            "market_value":    a.get("MKT_Total_Value"),
            "annual_tax":      a.get("AnnualTax"),
            "zoning":          safe(a.get("Zoning")),
            "neighborhood_cd": safe(a.get("NeighborhoodCode")),
            "routing":         routing,
            "legal_desc":      safe(a.get("mlegalDescription")),
        })

    print(f"Page {page+1}: {len(features)} records | Total so far: {len(records)}")
    if not data.get("exceededTransferLimit", False):
        break
    page += 1
    time.sleep(0.2)

with open(OUTPUT, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=[
        "township","parcel_id","owner_name","is_business",
        "property_addr","mailing_addr","acres","market_value","annual_tax",
        "zoning","neighborhood_cd","routing","legal_desc"
    ])
    writer.writeheader()
    writer.writerows(records)

print(f"\nDone. {len(records)} parcels written to: {OUTPUT}")
print(f"\nBreakdown by township:")
for twp in ["HUBBARD","FOWLER","VIENNA","UNKNOWN"]:
    count = sum(1 for r in records if r["township"] == twp)
    if count:
        print(f"  {twp}: {count}")

print("\nSAMPLE (first 10):")
for r in records[:10]:
    print(f"  {r['township']:<8} {r['parcel_id']:<12} {r['acres']:<8} {r['owner_name']:<35} {r['property_addr']}")

Page 1: 532 records | Total so far: 532

Done. 532 parcels written to: trumbull_4acre_parcels_final-Matt.csv

Breakdown by township:
  HUBBARD: 189
  FOWLER: 177
  VIENNA: 166

SAMPLE (first 10):
  HUBBARD  01-019300    6.96     DANIEL K ELLIOTT JR J/S             0 S MAIN
  HUBBARD  01-038900    5.01     TIMOTHY W BLAKE J/S                 2960 SCHOTTEN
  HUBBARD  01-063932    7.94     CAROL SYLVESTER                     0 SEIFERT LEWIS
  HUBBARD  01-066017    26.95    KATHRYN SARISKY J/S                 0 WICK CAMPBELL
  HUBBARD  01-071995    27.3364  BRANKO MARKULIN                     0 WICK CAMPBELL RD
  HUBBARD  01-137800    8.84     EDWARD J YASECHKO                   LONG
  HUBBARD  01-241170    9.86     WILLIAM N PATRICK J/S               E LIBERTY
  HUBBARD  01-259650    76.41    NICHOLAS V LIBEG TR                 4556 CATHERINE
  HUBBARD  01-260220    27.0848  R A A W LLC                         0 CHURCHILL HUBBARD RD
  HUBBARD  01-279500    12.39    MASSULLO FARMS &       

In [12]:
"""
Multi-County Campground / RV Park Parcel Query
Targets: Trumbull, Mahoning, Portage, Ashtabula (OH) + Mercer (PA)
DTE Codes: 415 = Trailer/Mobile Home Park  |  416 = Commercial Campground

Output: campgrounds_regional.csv

Requirements: pip install requests
"""

import requests, csv, json, time
from datetime import datetime

OUTPUT = f"campgrounds_regional_{datetime.now().strftime('%Y%m%d')}.csv"

# ── CANDIDATE FIELD NAMES for DTE/classification code (varies by county) ────
CLASSIFICATION_CANDIDATES = [
    "mClassificationId", "DTE_CODE", "DTE", "USE_CODE", "USECODE",
    "LUC", "LAND_USE_CODE", "CLASS_CODE", "PROPCLASS", "PROP_CLASS",
    "LANDUSECODE", "LandUseCode", "land_use", "PropertyClass"
]
OWNER_CANDIDATES    = ["OwnName", "OWNER", "OWNER1", "OWNER_NAME", "OWNERNAME", "DeededOwner"]
ACRES_CANDIDATES    = ["macres", "ACRES", "ASSR_ACRES", "ACREAGE", "GIS_ACRES", "CALC_ACRES"]
PARCEL_CANDIDATES   = ["PARCEL_ID", "PIN", "PARCELID", "APN", "PARCELNUMBER", "PARID", "ParcelID"]
ADDR_CANDIDATES     = ["mlocStrName", "SITEADDRESS", "SITE_ADDR", "ADDRESS", "ADDR", "SitusAddress"]
ADDRNO_CANDIDATES   = ["mlocStrNo", "SITESTREETNO", "SITE_NO", "STRNO", "HouseNumber"]
CITY_CANDIDATES     = ["mlocCity", "SITECITY", "CITY", "SITUS_CITY"]
MAIL_CANDIDATES     = ["TaxpStreetName", "MAIL_ADDR", "MAILADDR", "MAILSTREET"]
MAILNO_CANDIDATES   = ["TaxpStreetNumber", "MAIL_NO", "MAILNO"]
MAILCITY_CANDIDATES = ["TaxpCity", "MAIL_CITY", "MAILCITY"]
MAILST_CANDIDATES   = ["TaxpState", "MAIL_STATE", "MAILSTATE"]
MAILZIP_CANDIDATES  = ["TaxpZipcode", "MAIL_ZIP", "MAILZIP", "ZIPCODE"]
VALUE_CANDIDATES    = ["MKT_Total_Value", "TOTALVALUE", "MARKET_VALUE", "APPRTOTVAL", "APPR_VALUE"]

# ── COUNTY CONFIGURATIONS ────────────────────────────────────────────────────
COUNTIES = [
    {
        "name":    "TRUMBULL",
        "state":   "OH",
        "url":     "https://webgis.co.trumbull.oh.us/gisserver/rest/services/Maps/TaxParcels/MapServer/0",
        "deleted_field": "Par_Deleted",   # known — skip deleted parcels
        "deleted_value": "N",
    },
    {
        "name":    "MAHONING",
        "state":   "OH",
        # Try layer 0 first; AUDITOR_LOGIN service
        "url":     "https://gisapp.mahoningcountyoh.gov/arcgis/rest/services/AUDITOR_LOGIN/MapServer/0",
        "deleted_field": None,
        "deleted_value": None,
    },
    {
        "name":    "ASHTABULA",
        "state":   "OH",
        # FeatureServer with parcel export layer
        "url":     "https://parcelmap.ashtabulacounty.us/arcgis/rest/services/Parcel_Data/AshtabulaCounty_ParcelExports/FeatureServer/0",
        "deleted_field": None,
        "deleted_value": None,
    },
    {
        "name":    "PORTAGE",
        "state":   "OH",
        # Probe — Portage County uses ArcGIS Online hosted layers
        "url":     "https://services1.arcgis.com/nTkpEUMFHUNISlue/arcgis/rest/services/Portage_County_Parcels/FeatureServer/0",
        "deleted_field": None,
        "deleted_value": None,
    },
]

# Mercer County PA uses a separate system — see note at bottom of script.

CAMPGROUND_CODES = ["415", "416", "415 ", "416 "]  # include padded variants

# ── HELPERS ──────────────────────────────────────────────────────────────────
def safe(val):
    return str(val).strip() if val is not None else ""

def pick_field(available: dict, candidates: list) -> str | None:
    """Return the first candidate that exists in available fields (case-insensitive)."""
    upper = {k.upper(): v for k, v in available.items()}
    for c in candidates:
        if c.upper() in upper:
            return upper[c.upper()]
    return None

def probe_fields(base_url: str, session: requests.Session) -> tuple[dict, dict]:
    """
    Query one record to discover field names.
    Returns (field_map, raw_sample_attributes).
    """
    r = session.get(f"{base_url}/query", params={
        "where": "1=1", "outFields": "*",
        "resultRecordCount": 1, "returnGeometry": "false", "f": "json"
    }, timeout=20)
    r.raise_for_status()
    data = r.json()

    if "error" in data:
        raise RuntimeError(f"API error: {data['error']}")

    # Build name→name map (preserve actual casing)
    available = {f["name"].upper(): f["name"] for f in data.get("fields", [])}

    fields = {
        "classification": pick_field(available, CLASSIFICATION_CANDIDATES),
        "owner":          pick_field(available, OWNER_CANDIDATES),
        "acres":          pick_field(available, ACRES_CANDIDATES),
        "parcel":         pick_field(available, PARCEL_CANDIDATES),
        "addr":           pick_field(available, ADDR_CANDIDATES),
        "addrno":         pick_field(available, ADDRNO_CANDIDATES),
        "city":           pick_field(available, CITY_CANDIDATES),
        "mail":           pick_field(available, MAIL_CANDIDATES),
        "mailno":         pick_field(available, MAILNO_CANDIDATES),
        "mailcity":       pick_field(available, MAILCITY_CANDIDATES),
        "mailst":         pick_field(available, MAILST_CANDIDATES),
        "mailzip":        pick_field(available, MAILZIP_CANDIDATES),
        "value":          pick_field(available, VALUE_CANDIDATES),
    }

    sample = {}
    if data.get("features"):
        sample = data["features"][0].get("attributes", {})

    return fields, sample


def build_where(fields: dict, deleted_field: str, deleted_value: str) -> str | None:
    cf = fields.get("classification")
    if not cf:
        return None  # can't filter — will skip this county

    code_list = ", ".join(f"'{c}'" for c in CAMPGROUND_CODES)
    where = f"{cf} IN ({code_list})"

    if deleted_field and deleted_value:
        where += f" AND {deleted_field} = '{deleted_value}'"

    return where


def fetch_county(county: dict, session: requests.Session) -> list[dict]:
    name  = county["name"]
    url   = county["url"]
    print(f"\n{'─'*60}")
    print(f"  County: {name} ({county['state']})")
    print(f"  URL:    {url}")

    try:
        fields, sample = probe_fields(url, session)
        print(f"  Classification field: {fields['classification']}")
        print(f"  Owner field:          {fields['owner']}")
        print(f"  Acres field:          {fields['acres']}")

        if not fields["classification"]:
            print(f"  ⚠ No classification/DTE field found — skipping {name}")
            print(f"    Available fields in sample: {list(sample.keys())[:15]}")
            return []

        where = build_where(fields, county["deleted_field"], county["deleted_value"])
        print(f"  WHERE: {where}")

        out_fields = ",".join(v for v in fields.values() if v)

        records = []
        page = 0
        while True:
            params = {
                "where":             where,
                "outFields":         out_fields,
                "resultRecordCount": 1000,
                "resultOffset":      page * 1000,
                "returnGeometry":    "false",
                "f":                 "json"
            }
            r = session.get(f"{url}/query", params=params, timeout=30)
            data = r.json()

            if "error" in data:
                print(f"  ⚠ API error: {data['error']}")
                break

            features = data.get("features", [])
            if not features:
                break

            for feat in features:
                a = feat["attributes"]
                prop_addr = " ".join(filter(None, [
                    safe(a.get(fields["addrno"])),
                    safe(a.get(fields["addr"])),
                    safe(a.get(fields["city"])),
                ]))
                mail_addr = " ".join(filter(None, [
                    safe(a.get(fields["mailno"])),
                    safe(a.get(fields["mail"])),
                    safe(a.get(fields["mailcity"])),
                    safe(a.get(fields["mailst"])),
                    safe(a.get(fields["mailzip"])),
                ]))
                records.append({
                    "county":        name,
                    "state":         county["state"],
                    "parcel_id":     safe(a.get(fields["parcel"])),
                    "owner_name":    safe(a.get(fields["owner"])),
                    "dte_code":      safe(a.get(fields["classification"])),
                    "property_addr": prop_addr,
                    "mailing_addr":  mail_addr,
                    "acres":         a.get(fields["acres"]),
                    "market_value":  a.get(fields["value"]),
                })

            print(f"  Page {page+1}: {len(features)} | Running total: {len(records)}")
            if not data.get("exceededTransferLimit", False):
                break
            page += 1
            time.sleep(0.3)

        return records

    except Exception as e:
        print(f"  ✗ Failed: {e}")
        return []


# ── MAIN ─────────────────────────────────────────────────────────────────────
def main():
    session = requests.Session()
    session.headers.update({"User-Agent": "Regional-Campground-Query/1.0"})

    all_records = []

    for county in COUNTIES:
        records = fetch_county(county, session)
        all_records.extend(records)

    if not all_records:
        print("\n✗ No records returned from any county. Check endpoint accessibility.")
        return

    fieldnames = ["county","state","parcel_id","owner_name","dte_code",
                  "property_addr","mailing_addr","acres","market_value"]

    with open(OUTPUT, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    print(f"\n{'='*60}")
    print(f"Done. {len(all_records)} total campground/RV park parcels → {OUTPUT}")
    print(f"\nBreakdown by county:")
    for county in COUNTIES:
        count = sum(1 for r in all_records if r["county"] == county["name"])
        print(f"  {county['name']:<12} {count}")

    print(f"\nBreakdown by DTE code:")
    for code in ["415", "416"]:
        count = sum(1 for r in all_records if r["dte_code"].strip() == code)
        label = "Trailer/Mobile Home Park" if code == "415" else "Commercial Campground"
        print(f"  {code} ({label}): {count}")

    print(f"\nSample (first 10):")
    for r in all_records[:10]:
        print(f"  {r['county']:<12} {r['dte_code']}  {str(r['acres']):<8}  {r['owner_name']:<35}  {r['property_addr']}")


if __name__ == "__main__":
    main()

# ── MERCER COUNTY PA NOTE ────────────────────────────────────────────────────
"""
Mercer County PA does not use Ohio DTE codes. Pennsylvania uses a different
classification system. To find campgrounds in Mercer County PA:

Option 1 — Mercer County GIS portal:
  https://www.mcc.co.mercer.pa.us/346/GIS-Department
  Manual search by land use description containing "CAMP" or "RECREATION"

Option 2 — Pennsylvania Spatial Data Access (PASDA):
  https://www.pasda.psu.edu/
  Search: "Mercer County parcels" — downloadable shapefile with LUC field

Option 3 — Recreation.gov API (federal campgrounds only, free):
  https://ridb.recreation.gov/api/v1/campgrounds?state=PA&apikey=YOUR_KEY
  Sign up at: https://ridb.recreation.gov/

Option 4 — State of PA CAMA data (most complete):
  Contact Mercer County Assessment Office: (724) 662-3800
  Request export of parcels with land use codes containing "CAMP" or code 591
  PA uses LUC 591 = Campground/Recreational Vehicle Park
"""



────────────────────────────────────────────────────────────
  County: TRUMBULL (OH)
  URL:    https://webgis.co.trumbull.oh.us/gisserver/rest/services/Maps/TaxParcels/MapServer/0
  Classification field: mClassificationId
  Owner field:          OwnName
  Acres field:          macres
  WHERE: mClassificationId IN ('415', '416', '415 ', '416 ') AND Par_Deleted = 'N'
  Page 1: 117 | Running total: 117

────────────────────────────────────────────────────────────
  County: MAHONING (OH)
  URL:    https://gisapp.mahoningcountyoh.gov/arcgis/rest/services/AUDITOR_LOGIN/MapServer/0
  Classification field: None
  Owner field:          None
  Acres field:          ACRES
  ⚠ No classification/DTE field found — skipping MAHONING
    Available fields in sample: ['OBJECTID', 'TAX_MAP', 'HOLE_ID', 'ACRES', 'SQ_MILES', 'MUNICIP', 'Shape.area', 'Shape.len']

────────────────────────────────────────────────────────────
  County: ASHTABULA (OH)
  URL:    https://parcelmap.ashtabulacounty.us/arcgis/rest

'\nMercer County PA does not use Ohio DTE codes. Pennsylvania uses a different\nclassification system. To find campgrounds in Mercer County PA:\n\nOption 1 — Mercer County GIS portal:\n  https://www.mcc.co.mercer.pa.us/346/GIS-Department\n  Manual search by land use description containing "CAMP" or "RECREATION"\n\nOption 2 — Pennsylvania Spatial Data Access (PASDA):\n  https://www.pasda.psu.edu/\n  Search: "Mercer County parcels" — downloadable shapefile with LUC field\n\nOption 3 — Recreation.gov API (federal campgrounds only, free):\n  https://ridb.recreation.gov/api/v1/campgrounds?state=PA&apikey=YOUR_KEY\n  Sign up at: https://ridb.recreation.gov/\n\nOption 4 — State of PA CAMA data (most complete):\n  Contact Mercer County Assessment Office: (724) 662-3800\n  Request export of parcels with land use codes containing "CAMP" or code 591\n  PA uses LUC 591 = Campground/Recreational Vehicle Park\n'

In [13]:
"""
Multi-County Campground / RV Park Parcel Query
Targets: Trumbull, Mahoning, Portage, Ashtabula (OH) + Mercer (PA)
DTE Codes: 415 = Trailer/Mobile Home Park  |  416 = Commercial Campground

Output: campgrounds_regional.csv

Requirements: pip install requests
"""

import requests, csv, json, time
from datetime import datetime

OUTPUT = f"campgrounds_regional_{datetime.now().strftime('%Y%m%d')}.csv"

# ── CANDIDATE FIELD NAMES for DTE/classification code (varies by county) ────
CLASSIFICATION_CANDIDATES = [
    "mClassificationId", "DTE_CODE", "DTE", "USE_CODE", "USECODE",
    "LUC", "LAND_USE_CODE", "CLASS_CODE", "PROPCLASS", "PROP_CLASS",
    "LANDUSECODE", "LandUseCode", "land_use", "PropertyClass"
]
OWNER_CANDIDATES    = ["OwnName", "OWNER", "OWNER1", "OWNER_NAME", "OWNERNAME", "DeededOwner"]
ACRES_CANDIDATES    = ["macres", "ACRES", "ASSR_ACRES", "ACREAGE", "GIS_ACRES", "CALC_ACRES"]
PARCEL_CANDIDATES   = ["PARCEL_ID", "PIN", "PARCELID", "APN", "PARCELNUMBER", "PARID", "ParcelID"]
ADDR_CANDIDATES     = ["mlocStrName", "SITEADDRESS", "SITE_ADDR", "ADDRESS", "ADDR", "SitusAddress"]
ADDRNO_CANDIDATES   = ["mlocStrNo", "SITESTREETNO", "SITE_NO", "STRNO", "HouseNumber"]
CITY_CANDIDATES     = ["mlocCity", "SITECITY", "CITY", "SITUS_CITY"]
MAIL_CANDIDATES     = ["TaxpStreetName", "MAIL_ADDR", "MAILADDR", "MAILSTREET"]
MAILNO_CANDIDATES   = ["TaxpStreetNumber", "MAIL_NO", "MAILNO"]
MAILCITY_CANDIDATES = ["TaxpCity", "MAIL_CITY", "MAILCITY"]
MAILST_CANDIDATES   = ["TaxpState", "MAIL_STATE", "MAILSTATE"]
MAILZIP_CANDIDATES  = ["TaxpZipcode", "MAIL_ZIP", "MAILZIP", "ZIPCODE"]
VALUE_CANDIDATES    = ["MKT_Total_Value", "TOTALVALUE", "MARKET_VALUE", "APPRTOTVAL", "APPR_VALUE"]

# ── COUNTY CONFIGURATIONS ────────────────────────────────────────────────────
COUNTIES = [
    {
        "name":    "TRUMBULL",
        "state":   "OH",
        "url":     "https://webgis.co.trumbull.oh.us/gisserver/rest/services/Maps/TaxParcels/MapServer/0",
        "deleted_field": "Par_Deleted",   # known — skip deleted parcels
        "deleted_value": "N",
    },
    {
        "name":    "MAHONING",
        "state":   "OH",
        # Try layer 0 first; AUDITOR_LOGIN service
        "url":     "https://gisapp.mahoningcountyoh.gov/arcgis/rest/services/AUDITOR_LOGIN/MapServer/0",
        "deleted_field": None,
        "deleted_value": None,
    },
    {
        "name":    "ASHTABULA",
        "state":   "OH",
        # FeatureServer with parcel export layer
        "url":     "https://parcelmap.ashtabulacounty.us/arcgis/rest/services/Parcel_Data/AshtabulaCounty_ParcelExports/FeatureServer/0",
        "deleted_field": None,
        "deleted_value": None,
    },
    {
        "name":    "PORTAGE",
        "state":   "OH",
        # Probe — Portage County uses ArcGIS Online hosted layers
        "url":     "https://services1.arcgis.com/nTkpEUMFHUNISlue/arcgis/rest/services/Portage_County_Parcels/FeatureServer/0",
        "deleted_field": None,
        "deleted_value": None,
    },
]

# Mercer County PA uses a separate system — see note at bottom of script.

CAMPGROUND_CODES = ["415", "416", "415 ", "416 "]  # include padded variants

# ── HELPERS ──────────────────────────────────────────────────────────────────
def safe(val):
    return str(val).strip() if val is not None else ""

def pick_field(available: dict, candidates: list) -> str | None:
    """Return the first candidate that exists in available fields (case-insensitive)."""
    upper = {k.upper(): v for k, v in available.items()}
    for c in candidates:
        if c.upper() in upper:
            return upper[c.upper()]
    return None

def probe_fields(base_url: str, session: requests.Session) -> tuple[dict, dict]:
    """
    Query one record to discover field names.
    Returns (field_map, raw_sample_attributes).
    """
    r = session.get(f"{base_url}/query", params={
        "where": "1=1", "outFields": "*",
        "resultRecordCount": 1, "returnGeometry": "false", "f": "json"
    }, timeout=20)
    r.raise_for_status()
    data = r.json()

    if "error" in data:
        raise RuntimeError(f"API error: {data['error']}")

    # Build name→name map (preserve actual casing)
    available = {f["name"].upper(): f["name"] for f in data.get("fields", [])}

    fields = {
        "classification": pick_field(available, CLASSIFICATION_CANDIDATES),
        "owner":          pick_field(available, OWNER_CANDIDATES),
        "acres":          pick_field(available, ACRES_CANDIDATES),
        "parcel":         pick_field(available, PARCEL_CANDIDATES),
        "addr":           pick_field(available, ADDR_CANDIDATES),
        "addrno":         pick_field(available, ADDRNO_CANDIDATES),
        "city":           pick_field(available, CITY_CANDIDATES),
        "mail":           pick_field(available, MAIL_CANDIDATES),
        "mailno":         pick_field(available, MAILNO_CANDIDATES),
        "mailcity":       pick_field(available, MAILCITY_CANDIDATES),
        "mailst":         pick_field(available, MAILST_CANDIDATES),
        "mailzip":        pick_field(available, MAILZIP_CANDIDATES),
        "value":          pick_field(available, VALUE_CANDIDATES),
    }

    sample = {}
    if data.get("features"):
        sample = data["features"][0].get("attributes", {})

    return fields, sample


def build_where(fields: dict, deleted_field: str, deleted_value: str) -> str | None:
    cf = fields.get("classification")
    if not cf:
        return None  # can't filter — will skip this county

    code_list = ", ".join(f"'{c}'" for c in CAMPGROUND_CODES)
    where = f"{cf} IN ({code_list})"

    if deleted_field and deleted_value:
        where += f" AND {deleted_field} = '{deleted_value}'"

    return where


def fetch_county(county: dict, session: requests.Session) -> list[dict]:
    name  = county["name"]
    url   = county["url"]
    print(f"\n{'─'*60}")
    print(f"  County: {name} ({county['state']})")
    print(f"  URL:    {url}")

    try:
        fields, sample = probe_fields(url, session)
        print(f"  Classification field: {fields['classification']}")
        print(f"  Owner field:          {fields['owner']}")
        print(f"  Acres field:          {fields['acres']}")

        if not fields["classification"]:
            print(f"  ⚠ No classification/DTE field found — skipping {name}")
            print(f"    Available fields in sample: {list(sample.keys())[:15]}")
            return []

        where = build_where(fields, county["deleted_field"], county["deleted_value"])
        print(f"  WHERE: {where}")

        out_fields = ",".join(v for v in fields.values() if v)

        records = []
        page = 0
        while True:
            params = {
                "where":             where,
                "outFields":         out_fields,
                "resultRecordCount": 1000,
                "resultOffset":      page * 1000,
                "returnGeometry":    "false",
                "f":                 "json"
            }
            r = session.get(f"{url}/query", params=params, timeout=30)
            data = r.json()

            if "error" in data:
                print(f"  ⚠ API error: {data['error']}")
                break

            features = data.get("features", [])
            if not features:
                break

            for feat in features:
                a = feat["attributes"]
                prop_addr = " ".join(filter(None, [
                    safe(a.get(fields["addrno"])),
                    safe(a.get(fields["addr"])),
                    safe(a.get(fields["city"])),
                ]))
                mail_addr = " ".join(filter(None, [
                    safe(a.get(fields["mailno"])),
                    safe(a.get(fields["mail"])),
                    safe(a.get(fields["mailcity"])),
                    safe(a.get(fields["mailst"])),
                    safe(a.get(fields["mailzip"])),
                ]))
                records.append({
                    "county":        name,
                    "state":         county["state"],
                    "parcel_id":     safe(a.get(fields["parcel"])),
                    "owner_name":    safe(a.get(fields["owner"])),
                    "dte_code":      safe(a.get(fields["classification"])),
                    "property_addr": prop_addr,
                    "mailing_addr":  mail_addr,
                    "acres":         a.get(fields["acres"]),
                    "market_value":  a.get(fields["value"]),
                })

            print(f"  Page {page+1}: {len(features)} | Running total: {len(records)}")
            if not data.get("exceededTransferLimit", False):
                break
            page += 1
            time.sleep(0.3)

        return records

    except Exception as e:
        print(f"  ✗ Failed: {e}")
        return []


# ── MAIN ─────────────────────────────────────────────────────────────────────
def main():
    session = requests.Session()
    session.headers.update({"User-Agent": "Regional-Campground-Query/1.0"})

    all_records = []

    for county in COUNTIES:
        records = fetch_county(county, session)
        all_records.extend(records)

    if not all_records:
        print("\n✗ No records returned from any county. Check endpoint accessibility.")
        return

    fieldnames = ["county","state","parcel_id","owner_name","dte_code",
                  "property_addr","mailing_addr","acres","market_value"]

    with open(OUTPUT, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_records)

    print(f"\n{'='*60}")
    print(f"Done. {len(all_records)} total campground/RV park parcels → {OUTPUT}")
    print(f"\nBreakdown by county:")
    for county in COUNTIES:
        count = sum(1 for r in all_records if r["county"] == county["name"])
        print(f"  {county['name']:<12} {count}")

    print(f"\nBreakdown by DTE code:")
    for code in ["415", "416"]:
        count = sum(1 for r in all_records if r["dte_code"].strip() == code)
        label = "Trailer/Mobile Home Park" if code == "415" else "Commercial Campground"
        print(f"  {code} ({label}): {count}")

    print(f"\nSample (first 10):")
    for r in all_records[:10]:
        print(f"  {r['county']:<12} {r['dte_code']}  {str(r['acres']):<8}  {r['owner_name']:<35}  {r['property_addr']}")


if __name__ == "__main__":
    main()


────────────────────────────────────────────────────────────
  County: TRUMBULL (OH)
  URL:    https://webgis.co.trumbull.oh.us/gisserver/rest/services/Maps/TaxParcels/MapServer/0
  Classification field: mClassificationId
  Owner field:          OwnName
  Acres field:          macres
  WHERE: mClassificationId IN ('415', '416', '415 ', '416 ') AND Par_Deleted = 'N'
  Page 1: 117 | Running total: 117

────────────────────────────────────────────────────────────
  County: MAHONING (OH)
  URL:    https://gisapp.mahoningcountyoh.gov/arcgis/rest/services/AUDITOR_LOGIN/MapServer/0
  Classification field: None
  Owner field:          None
  Acres field:          ACRES
  ⚠ No classification/DTE field found — skipping MAHONING
    Available fields in sample: ['OBJECTID', 'TAX_MAP', 'HOLE_ID', 'ACRES', 'SQ_MILES', 'MUNICIP', 'Shape.area', 'Shape.len']

────────────────────────────────────────────────────────────
  County: ASHTABULA (OH)
  URL:    https://parcelmap.ashtabulacounty.us/arcgis/rest

In [14]:
import requests, json

session = requests.Session()

# ── MAHONING: find which layer has parcel/classification data ──
print("=== MAHONING LAYERS ===")
r = session.get("https://gisapp.mahoningcountyoh.gov/arcgis/rest/services/AUDITOR_LOGIN/MapServer?f=json", timeout=15)
data = r.json()
for layer in data.get("layers", []):
    print(f"  ID:{layer['id']}  {layer['name']}")

# ── ASHTABULA: try the MapServer instead of FeatureServer ──
print("\n=== ASHTABULA LAYERS ===")
r = session.get("https://parcelmap.ashtabulacounty.us/arcgis/rest/services/Online_Parcel/MapServer?f=json", timeout=20)
data = r.json()
for layer in data.get("layers", []):
    print(f"  ID:{layer['id']}  {layer['name']}")

# ── PORTAGE: probe the ArcGIS Online org ──
print("\n=== PORTAGE: search for parcel service ===")
r = session.get("https://portagecounty.maps.arcgis.com/sharing/rest/search?q=parcels&num=5&f=json", timeout=15)
data = r.json()
for item in data.get("results", []):
    print(f"  {item.get('title')}  |  {item.get('url')}")


=== MAHONING LAYERS ===
  ID:0  Property Holes
  ID:1  Property Hole Labels
  ID:2  Addresses Mismatch
  ID:3  911 Addresses Labels
  ID:4  MVP Addresses Labels

=== ASHTABULA LAYERS ===
  ID:0  Parcels_Online
  ID:1  Parcel Location Address
  ID:2  Acreage
  ID:3  Owner
  ID:4  Conveyance Division
  ID:5  Road Vacations
  ID:6  Encumbrance
  ID:7  Right of Way
  ID:8  All Parcel Line
  ID:9  All Parcel Label

=== PORTAGE: search for parcel service ===
  Regrid USA Nationwide Parcel Boundaries  |  https://tiles.arcgis.com/tiles/KzeiCaQsMoeCfoCq/arcgis/rest/services/Regrid_Nationwide_Parcel_Boundaries_v1/MapServer
  North Carolina Parcels (Polygons)  |  https://services.nconemap.gov/secure/rest/services/NC1Map_Parcels/MapServer/1
  Connecticut CAMA and Parcel Layer  |  https://services3.arcgis.com/3FL1kr7L4LvwA2Kb/arcgis/rest/services/Connecticut_CAMA_and_Parcel_Layer/FeatureServer
  Maine Parcels Organized Towns Feature  |  https://services1.arcgis.com/RbMX0mRVOFNTdLzd/arcgis/rest/serv

In [15]:
import requests

session = requests.Session()

# ── ASHTABULA: probe layer 0 for field names ──
print("=== ASHTABULA LAYER 0 FIELDS ===")
r = session.get(
    "https://parcelmap.ashtabulacounty.us/arcgis/rest/services/Online_Parcel/MapServer/0/query",
    params={"where":"1=1","outFields":"*","resultRecordCount":1,"returnGeometry":"false","f":"json"},
    timeout=20
)
data = r.json()
print("Fields:", [f["name"] for f in data.get("fields",[])])
if data.get("features"):
    import json
    print("Sample:", json.dumps(data["features"][0]["attributes"], indent=2)[:1500])

# ── MAHONING: find the real parcel service ──
print("\n=== MAHONING: all services on server ===")
r = session.get("https://gisapp.mahoningcountyoh.gov/arcgis/rest/services?f=json", timeout=15)
data = r.json()
for svc in data.get("services", []):
    print(f"  {svc['name']}  ({svc['type']})")
for folder in data.get("folders", []):
    print(f"  [folder] {folder}")

# ── PORTAGE: try Beacon/Schneider which serves Portage ──
print("\n=== PORTAGE: probe Schneider ===")
r = session.get(
    "https://beacon.schneidercorp.com/api/parcels/search",
    params={"county":"Portage","state":"OH","dte":"416"},
    timeout=10
)
print(f"  Status: {r.status_code}")

# ── PORTAGE: try direct GIS server ──
print("\n=== PORTAGE: probe county GIS server ===")
for url in [
    "https://gis.co.portage.oh.us/arcgis/rest/services?f=json",
    "https://portage.maps.arcgis.com/sharing/rest/search?q=parcel+type:Feature+Service&f=json&num=5",
]:
    try:
        r = session.get(url, timeout=10)
        print(f"  {url}")
        print(f"  Status: {r.status_code}  Length: {len(r.text)}")
        if r.status_code == 200 and len(r.text) > 50:
            print(f"  Preview: {r.text[:300]}")
    except Exception as e:
        print(f"  {url} → {e}")

=== ASHTABULA LAYER 0 FIELDS ===
Fields: ['OBJECTID', 'Name', 'Type', 'StatedArea', 'Historical', 'SystemStartDate', 'SystemEndDate', 'PIN', 'ConveyanceType', 'ConveyanceDesignator', 'SimConDivType', 'BlockDesignator', 'EncumbranceType', 'SeparatedRightType', 'HistoryType', 'FloorDesignator', 'PrincipalMeridian', 'TownshipNumber', 'TownshipFraction', 'TownshipDirection', 'RangeNumber', 'RangeFraction', 'RangeDirection', 'TownshipDupCode', 'PLSSID', 'FirstDivisionType', 'FirstDivisionDupCode', 'SecondDivisionNumber', 'SecondDivisionSuffix', 'SecondDivisionType', 'SpecialSurveyNumber', 'SpecialSurveySuffix', 'SpecialSurveyType', 'SpecialSurveyNotes', 'CreatedBy', 'ModifiedBy', 'ModifyDate', 'Encumbrance', 'Name_Encumbrance', 'jur', 'Grantor', 'Granted_To', 'Use_of', 'Vol___Page', 'Date', 'Survey', 'Sublot_num', 'UNIT_NUM', 'FLOOR_NUM', 'SUBLOT_NUMBER', 'PLAT_VOL_PAGE', 'DECLAR_VOL_PAGE', 'PIN_NO_DASHES', 'Acreage', 'Calculated_Acreage', 'Location_Address', 'Owner_Name', 'Owner_Address', 

In [16]:
import requests, json

session = requests.Session()

# ── MAHONING: probe the two most likely services ──
for svc in ["PUBLIC_WEBSITE_SQLDATA", "PUBLIC_WEBSITE_CADASTRAL"]:
    print(f"\n=== MAHONING {svc} LAYERS ===")
    try:
        r = session.get(
            f"https://gisapp.mahoningcountyoh.gov/arcgis/rest/services/{svc}/MapServer?f=json",
            timeout=15
        )
        data = r.json()
        for layer in data.get("layers", []):
            print(f"  ID:{layer['id']}  {layer['name']}")
    except Exception as e:
        print(f"  Error: {e}")

# ── MAHONING SQLDATA: probe layer 0 fields ──
print("\n=== MAHONING SQLDATA LAYER 0 SAMPLE ===")
try:
    r = session.get(
        "https://gisapp.mahoningcountyoh.gov/arcgis/rest/services/PUBLIC_WEBSITE_SQLDATA/MapServer/0/query",
        params={"where":"1=1","outFields":"*","resultRecordCount":1,"returnGeometry":"false","f":"json"},
        timeout=15
    )
    data = r.json()
    print("Fields:", [f["name"] for f in data.get("fields",[])])
    if data.get("features"):
        print("Sample:", json.dumps(data["features"][0]["attributes"], indent=2)[:2000])
except Exception as e:
    print(f"  Error: {e}")

# ── ASHTABULA: confirm Landuse_Code values for campgrounds ──
print("\n=== ASHTABULA: campground code check ===")
r = session.get(
    "https://parcelmap.ashtabulacounty.us/arcgis/rest/services/Online_Parcel/MapServer/0/query",
    params={
        "where": "Landuse_Code IN ('415','416','415 ','416 ')",
        "outFields": "PIN,Owner_Name,Acreage,Location_Address,Owner_Address,Owner_City,Landuse_Code",
        "resultRecordCount": 5,
        "returnGeometry": "false",
        "f": "json"
    },
    timeout=20
)
data = r.json()
print(f"  Result count: {len(data.get('features',[]))}")
for feat in data.get("features", []):
    a = feat["attributes"]
    print(f"  {a.get('Landuse_Code')}  {a.get('Acreage')}ac  {a.get('Owner_Name')}  {a.get('Location_Address')}")

# ── PORTAGE: try Beacon GIS direct ──
print("\n=== PORTAGE: Beacon GIS probe ===")
r = session.get(
    "https://beacon.schneidercor

SyntaxError: unterminated string literal (detected at line 56) (599921743.py, line 56)

In [17]:
import requests, json

session = requests.Session()

# ── MAHONING: probe the two most likely services ──
for svc in ["PUBLIC_WEBSITE_SQLDATA", "PUBLIC_WEBSITE_CADASTRAL"]:
    print(f"\n=== MAHONING {svc} LAYERS ===")
    try:
        r = session.get(
            f"https://gisapp.mahoningcountyoh.gov/arcgis/rest/services/{svc}/MapServer?f=json",
            timeout=15
        )
        data = r.json()
        for layer in data.get("layers", []):
            print(f"  ID:{layer['id']}  {layer['name']}")
    except Exception as e:
        print(f"  Error: {e}")

# ── MAHONING SQLDATA: probe layer 0 fields ──
print("\n=== MAHONING SQLDATA LAYER 0 SAMPLE ===")
try:
    r = session.get(
        "https://gisapp.mahoningcountyoh.gov/arcgis/rest/services/PUBLIC_WEBSITE_SQLDATA/MapServer/0/query",
        params={"where":"1=1","outFields":"*","resultRecordCount":1,"returnGeometry":"false","f":"json"},
        timeout=15
    )
    data = r.json()
    print("Fields:", [f["name"] for f in data.get("fields",[])])
    if data.get("features"):
        print("Sample:", json.dumps(data["features"][0]["attributes"], indent=2)[:2000])
except Exception as e:
    print(f"  Error: {e}")

# ── ASHTABULA: confirm Landuse_Code values for campgrounds ──
print("\n=== ASHTABULA: campground code check ===")
r = session.get(
    "https://parcelmap.ashtabulacounty.us/arcgis/rest/services/Online_Parcel/MapServer/0/query",
    params={
        "where": "Landuse_Code IN ('415','416','415 ','416 ')",
        "outFields": "PIN,Owner_Name,Acreage,Location_Address,Owner_Address,Owner_City,Landuse_Code",
        "resultRecordCount": 5,
        "returnGeometry": "false",
        "f": "json"
    },
    timeout=20
)
data = r.json()
print(f"  Result count: {len(data.get('features',[]))}")
for feat in data.get("features", []):
    a = feat["attributes"]
    print(f"  {a.get('Landuse_Code')}  {a.get('Acreage')}ac  {a.get('Owner_Name')}  {a.get('Location_Address')}")

# ── PORTAGE: try Beacon GIS direct ──
print("\n=== PORTAGE: Beacon GIS probe ===")
r = session.get(
    "https://beacon.schneidercorp.com/Application.aspx?AppID=768&LayerID=12967&PageTypeID=2&PageID=6222",
    timeout=10
)
print(f"  Status: {r.status_code}")


=== MAHONING PUBLIC_WEBSITE_SQLDATA LAYERS ===
  ID:0  PROPERTIES
  ID:1  RETIRED PROPERTIES
  ID:2  NEIGHBORHOODS
  ID:3  MUNICIPAL DISTRICTS
  ID:4  SUBDIVISIONS
  ID:5  CEMETERIES
  ID:6  PARKS
  ID:7  ZONING

=== MAHONING PUBLIC_WEBSITE_CADASTRAL LAYERS ===
  ID:0  PROPERTY LINES
  ID:1  PROPERTIES
  ID:2  PROPERTY BOUNDARIES
  ID:3  TAX DISTRICT BOUNDARIES
  ID:4  TAX DISTRICT LABELS
  ID:5  RETIRED PROPERTIES
  ID:6  RETIRED PROPERTY BOUNDARIES
  ID:7  RETIRED PROPERTY LABELS
  ID:8  PARCEL ID
  ID:9  TEXT
  ID:10  LOT ID
  ID:11  TEXT
  ID:12  PARCEL DIMENSIONS
  ID:13  TEXT
  ID:14  ROAD FRONTAGE
  ID:15  TEXT
  ID:16  MISC TEXT
  ID:17  TEXT
  ID:18  SUBDIVISIONS
  ID:19  SUBDIVISION BOUNDARIES
  ID:20  SUBDIVISION LABELS
  ID:21  CEMETERIES
  ID:22  CEMETERY BOUNDARIES
  ID:23  CEMETERY LABELS
  ID:24  PARKS
  ID:25  PARK BOUNDARIES
  ID:26  PARK LABELS
  ID:27  NEIGHBORHOODS
  ID:28  UNKNOWN PROPERTIES
  ID:29  UNKNOWN BOUNDARIES
  ID:30  UNKNOWN LABELS
  ID:31  VACATED PRO

In [18]:
import requests, csv, time
from datetime import datetime

OUTPUT  = f"campgrounds_regional_{datetime.now().strftime('%Y%m%d')}.csv"
CODES   = ("'415','416','415 ','416 '")

def safe(val):
    return str(val).strip() if val is not None else ""

def fetch(session, county, state, url, where, fields_map, deleted=None):
    records, page = [], 0
    out_fields = ",".join(v for v in fields_map.values() if v)
    full_where = where + (f" AND {deleted}" if deleted else "")
    print(f"\n── {county} ({state})")
    print(f"   WHERE: {full_where}")
    while True:
        params = {
            "where": full_where, "outFields": out_fields,
            "resultRecordCount": 1000, "resultOffset": page * 1000,
            "returnGeometry": "false", "f": "json"
        }
        r = session.get(f"{url}/query", params=params, timeout=30)
        data = r.json()
        if "error" in data:
            print(f"   ✗ API error: {data['error']}")
            break
        features = data.get("features", [])
        if not features:
            break
        for feat in features:
            a = feat["attributes"]
            records.append({
                "county":        county,
                "state":         state,
                "parcel_id":     safe(a.get(fields_map["parcel"])),
                "owner_name":    safe(a.get(fields_map["owner"])),
                "dte_code":      safe(a.get(fields_map["dte"])),
                "property_addr": " ".join(filter(None,[
                                    safe(a.get(fields_map["addrno"])),
                                    safe(a.get(fields_map["addr"])),
                                    safe(a.get(fields_map["city"])),
                                 ])),
                "mailing_addr":  safe(a.get(fields_map.get("mail",""))),
                "acres":         a.get(fields_map["acres"]),
                "market_value":  a.get(fields_map.get("value")),
            })
        print(f"   Page {page+1}: {len(features)} | Total: {len(records)}")
        if not data.get("exceededTransferLimit", False):
            break
        page += 1
        time.sleep(0.2)
    return records

session = requests.Session()
all_records = []

# ── TRUMBULL ──────────────────────────────────────────────────────────────────
all_records += fetch(
    session, "TRUMBULL", "OH",
    url = "https://webgis.co.trumbull.oh.us/gisserver/rest/services/Maps/TaxParcels/MapServer/0",
    where = f"mClassificationId IN ({CODES})",
    fields_map = {
        "parcel":  "PARCEL_ID",
        "owner":   "OwnName",
        "dte":     "mClassificationId",
        "addrno":  "mlocStrNo",
        "addr":    "mlocStrName",
        "city":    "mlocCity",
        "mail":    "TaxpCity",
        "acres":   "macres",
        "value":   "MKT_Total_Value",
    },
    deleted = "Par_Deleted = 'N'"
)

# ── MAHONING ──────────────────────────────────────────────────────────────────
all_records += fetch(
    session, "MAHONING", "OH",
    url = "https://gisapp.mahoningcountyoh.gov/arcgis/rest/services/PUBLIC_WEBSITE_SQLDATA/MapServer/0",
    where = f"LANDUSE IN ({CODES})",
    fields_map = {
        "parcel":  "PARCEL_ID",
        "owner":   "OWNNAME3",
        "dte":     "LANDUSE",
        "addrno":  "LOCNUM",
        "addr":    "LOCSTREET",
        "city":    "MUNICIP",
        "mail":    "MUNICIP",
        "acres":   "ACRES",
        "value":   "TOTALMARKET",
    }
)

# ── ASHTABULA ─────────────────────────────────────────────────────────────────
all_records += fetch(
    session, "ASHTABULA", "OH",
    url = "https://parcelmap.ashtabulacounty.us/arcgis/rest/services/Online_Parcel/MapServer/0",
    where = f"Landuse_Code IN ({CODES})",
    fields_map = {
        "parcel":  "PIN",
        "owner":   "Owner_Name",
        "dte":     "Landuse_Code",
        "addrno":  "",
        "addr":    "Location_Address",
        "city":    "Owner_City",
        "mail":    "Owner_Address",
        "acres":   "Acreage",
        "value":   None,
    }
)

# ── WRITE CSV ─────────────────────────────────────────────────────────────────
fieldnames = ["county","state","parcel_id","owner_name","dte_code",
              "property_addr","mailing_addr","acres","market_value"]

with open(OUTPUT, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(all_records)

print(f"\n{'='*60}")
print(f"Done. {len(all_records)} records → {OUTPUT}")
print("\nBreakdown:")
for county in ["TRUMBULL","MAHONING","ASHTABULA"]:
    n = sum(1 for r in all_records if r["county"] == county)
    print(f"  {county:<12} {n}")
for code, label in [("415","Mobile Home/Trailer Park"),("416","Commercial Campground")]:
    n = sum(1 for r in all_records if r["dte_code"].strip() == code)
    print(f"  DTE {code} ({label}): {n}")

print("\nSample (first 10):")
for r in all_records[:10]:
    print(f"  {r['county']:<12} {r['dte_code']}  {str(r['acres'] or ''):<8}  {r['owner_name']:<35}  {r['property_addr']}")


── TRUMBULL (OH)
   WHERE: mClassificationId IN ('415','416','415 ','416 ') AND Par_Deleted = 'N'
   Page 1: 117 | Total: 117

── MAHONING (OH)
   WHERE: LANDUSE IN ('415','416','415 ','416 ')
   ✗ API error: {'code': 400, 'message': 'Failed to execute query.', 'details': []}

── ASHTABULA (OH)
   WHERE: Landuse_Code IN ('415','416','415 ','416 ')
   Page 1: 126 | Total: 126

Done. 243 records → campgrounds_regional_20260410.csv

Breakdown:
  TRUMBULL     117
  MAHONING     0
  ASHTABULA    126
  DTE 415 (Mobile Home/Trailer Park): 154
  DTE 416 (Commercial Campground): 89

Sample (first 10):
  TRUMBULL     415  10.12     LEO J CHIARULLO TRUSTEE              5029 PARKMAN
  TRUMBULL     415  7.67      WARREN WOODS ESTATES                 180 NORTH RIVER
  TRUMBULL     415  4.78      HIGHFALUTIN MHP 001                  2018 S TOD
  TRUMBULL     415  83.82     IMPERIAL COMMUNITIES INC             2382 HALLOCK YOUNG
  TRUMBULL     415  83.82     IMPERIAL COMMUNITIES INC             2382 

In [19]:
import requests, json

session = requests.Session()

# ── MAHONING: check if LANDUSE is numeric and sample its values ──
print("=== MAHONING: LANDUSE field sample ===")
r = session.get(
    "https://gisapp.mahoningcountyoh.gov/arcgis/rest/services/PUBLIC_WEBSITE_SQLDATA/MapServer/0/query",
    params={
        "where": "1=1",
        "outFields": "PARCELID,OWNER,LANDUSE,ACREAGE,SITEADDRESS",
        "resultRecordCount": 5,
        "returnGeometry": "false",
        "f": "json"
    },
    timeout=15
)
data = r.json()
if data.get("features"):
    for feat in data["features"]:
        a = feat["attributes"]
        print(f"  LANDUSE={repr(a.get('LANDUSE'))}  ACRES={a.get('ACREAGE')}  {a.get('OWNER')}")

# Try numeric codes
print("\n=== MAHONING: numeric code query (415/416) ===")
for where in [
    "LANDUSE = 416",
    "LANDUSE = '416'",
    "LANDUSE LIKE '%416%'",
    "LANDUSE LIKE '%CAMP%'",
]:
    r = session.get(
        "https://gisapp.mahoningcountyoh.gov/arcgis/rest/services/PUBLIC_WEBSITE_SQLDATA/MapServer/0/query",
        params={"where": where, "outFields": "PARCELID,OWNER,LANDUSE,ACREAGE,SITEADDRESS",
                "resultRecordCount": 3, "returnGeometry": "false", "f": "json"},
        timeout=15
    )
    data = r.json()
    count = len(data.get("features", []))
    err = data.get("error", {}).get("message", "")
    print(f"  WHERE '{where}' → {count} results  {err}")
    for feat in data.get("features", []):
        a = feat["attributes"]
        print(f"    LANDUSE={repr(a.get('LANDUSE'))}  {a.get('OWNER')}  {a.get('SITEADDRESS')}")

# ── PORTAGE: try the county auditor's Beacon instance directly ──
print("\n=== PORTAGE: auditor site check ===")
for url in [
    "https://gis.portagecounty.us/arcgis/rest/services?f=json",
    "https://portagecountyauditor.org/gis",
    "https://www.portagecounty.us/auditor",
]:
    try:
        r = session.get(url, timeout=8)
        print(f"  {url}  →  {r.status_code}  ({len(r.text)} bytes)")
    except Exception as e:
        print(f"  {url}  →  {e}")

=== MAHONING: LANDUSE field sample ===

=== MAHONING: numeric code query (415/416) ===
  WHERE 'LANDUSE = 416' → 0 results  Failed to execute query.
  WHERE 'LANDUSE = '416'' → 0 results  Failed to execute query.
  WHERE 'LANDUSE LIKE '%416%'' → 0 results  Failed to execute query.
  WHERE 'LANDUSE LIKE '%CAMP%'' → 0 results  Failed to execute query.

=== PORTAGE: auditor site check ===
  https://gis.portagecounty.us/arcgis/rest/services?f=json  →  HTTPSConnectionPool(host='gis.portagecounty.us', port=443): Max retries exceeded with url: /arcgis/rest/services?f=json (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7ac3aa98c8f0>: Failed to resolve 'gis.portagecounty.us' ([Errno -2] Name or service not known)"))
  https://portagecountyauditor.org/gis  →  404  (310 bytes)
  https://www.portagecounty.us/auditor  →  HTTPSConnectionPool(host='www.portagecounty.us', port=443): Max retries exceeded with url: /auditor (Caused by NameResolutionError("<urllib3.connec